## 1. Module Import
The below listed modules are nessesary for various purposes
1. `os` - for using various os related funtionalities such as `makedir`
2. `Path` - for retriving files present inside other directory
3. `pyedflib` - for converting the numpy files into edf
4. `numpy` - for numerical calculations on edf files
5. `mne` - for reading the edf files
6. `gc` - for collecting and deleting garbage memory

In [1]:
import os
from pathlib import Path
import pyedflib
import numpy as np
import mne
import gc

## 1. Data Analysis
### 1.1 Looking at CHB-MIT Data
For better understanding of the data and visualisation of data using the `mne` module, reading one instance of CHB-MIT dataset

In [2]:
import mne
# Load one EDF file and inspect it
chb_raw = mne.io.read_raw_edf('../data/raw/CHB-MIT/chb01/chb01_01.edf', preload=True)
print(chb_raw.info)          # channels, sampling rate, duration
chb_channel_names = chb_raw.ch_names
print(chb_raw.ch_names)      # channel names
dump_extra_plot = chb_raw.plot(proj=False)               # visualize the raw signal

FileNotFoundError: File does not exist: "d:\Projects\Mini & Major Project\eeg_hdc\src\..\data\raw\CHB-MIT\chb01\chb01_01.edf"

### 1.2 Looking at Seina Scalp Dataset
Continuing the data-analysis and visualisation by observing one instance from the SienaScalp EEG dataset.

In [ ]:
import mne

# Load one EDF file and inspect it
seina_raw = mne.io.read_raw_edf('../data/raw/SienaScalp/PN00/PN00-1.edf', preload=True)
print(seina_raw.info)          # channels, sampling rate, duration
seina_channel_names = seina_raw.ch_names
print(seina_raw.ch_names)      # channel names
dump_extra_plot = seina_raw.plot(proj=False)               # visualize the raw signal

### 1.3 Printing channels
Checking for common channels in the dataset for further processing.

In [ ]:
print(seina_channel_names)
print(chb_channel_names)

# 2. Preprocessing
## 2.1 Defining Values

In [ ]:
common_sampling_frequency = 256

siena_eeg_channels = [
    'Fp1', 'F3', 'C3', 'P3', 'O1', 'F7', 'T3', 'T5',
    'Fc1', 'Fc5', 'Cp1', 'Cp5', 'F9', 'Fz', 'Cz', 'Pz',
    'Fp2', 'F4', 'C4', 'P4', 'O2', 'F8', 'T4', 'T6',
    'Fc2', 'Fc6', 'Cp2', 'Cp6', 'F10'
]

chb_eeg_required_channels = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ",
    "P7-T7", "T8-P8",
]

standard_to_siena_mapping = {
    "T7": "T3",
    "P7": "T5",
    "T8": "T4",
    "P8": "T6",
}

bipolar_to_monopolar_mapping = {
    "FP1-F7": ("Fp1", "F7"),
    "F7-T7": ("F7", "T3"),
    "T7-P7": ("T3", "T5"),
    "P7-O1": ("T5", "O1"),

    "FP1-F3": ("Fp1", "F3"),
    "F3-C3": ("F3", "C3"),
    "C3-P3": ("C3", "P3"),
    "P3-O1": ("P3", "O1"),

    "FP2-F4": ("Fp2", "F4"),
    "F4-C4": ("F4", "C4"),
    "C4-P4": ("C4", "P4"),
    "P4-O2": ("P4", "O2"),

    "FP2-F8": ("Fp2", "F8"),
    "F8-T8": ("F8", "T4"),
    "T8-P8": ("T4", "T6"),
    "P8-O2": ("T6", "O2"),

    "FZ-CZ": ("Fz", "Cz"),
    "CZ-PZ": ("Cz", "Pz"),

    "P7-T7": ("T5", "T3"),
}


In [ ]:
import pyedflib
from pathlib import Path

def inspect_unknown_channels(edf_path, target_labels=("1", "2", "MK")):
    f = pyedflib.EdfReader(str(edf_path))
    labels = f.getSignalLabels()
    report = {}
    for label in target_labels:
        if label in labels:
            idx = labels.index(label)
            report[label] = {
                "physical_dimension": f.getPhysicalDimension(idx),
                "transducer": f.getTransducer(idx),
                "sample_rate": f.getSampleFrequency(idx),
                "prefilter": f.getPrefilter(idx),
                "physical_min": f.getPhysicalMinimum(idx),
                "physical_max": f.getPhysicalMaximum(idx),
            }
    f.close()
    return report

# Check across multiple subjects, not just one — naming/meaning may differ per file
siena_root = Path("../data/raw/SienaScalp")
for subject_dir in sorted(siena_root.iterdir()):
    for edf_file in sorted(subject_dir.glob("*.edf")):
        result = inspect_unknown_channels(edf_file)
        if result:
            print(edf_file.name, result)